In [477]:
import numpy as np
import sisl
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from sisl import viz
from scipy.spatial import cKDTree

# Automation using `kind = 'armchair'`

In [ ]:
BOND = 1.42  # C-C bond length in Angstrom
KIND = 'armchair'  # 'armchair' or 'zigzag'
VACUUM = 3.0  # Vacuum layer in Angstrom
WIDTH = 3  # Width of the nanoribbon in number of dimer lines
LENGTH = 6  # Length of the nanoribbon in number of unit cells
# ---------------------------
# Geometry helper functions
# ---------------------------

def get_coordinates(structure):
    """Return atomic coordinates as numpy array."""
    if isinstance(structure, sisl.Geometry):
        return structure.xyz
    elif isinstance(structure, np.ndarray):
        return structure
    else:
        raise TypeError("Input must be a sisl.Geometry or numpy.ndarray.")
    
def find_nearest_atoms(structure, center, n=6):
    """Return indices of n atoms closest to a given center."""
    coords = get_coordinates(structure)
    distances = np.linalg.norm(coords - center, axis=1)
    return np.argsort(distances)[:n]


def guess_hexagon_center(structure):
    """
    Try to locate the center of a hexagon near the geometric center.
    If not found, iteratively shift the center guess.
    """
    coords = get_coordinates(structure)
    center = coords.mean(axis=0) 
    NN_2_DIST = 2 * BOND * np.cos(np.deg2rad(30))  # distance between two second-nearest neighbors in graphene
    RTOL = 0.05  # relative tolerance for checking if center lies on bonds
    
    atom_idx = find_nearest_atoms(structure, center, n=6)
    hex_coords = coords[atom_idx]
    center = hex_coords.mean(axis=0)
    atom1, atom2 = hex_coords[[0, 1]]
    if np.isclose(atom1[1], atom2[1], rtol=RTOL) and np.isclose(atom1[1], center[1], rtol=RTOL):
        print("Warning: The geometric center lies on bonds. Trying to shift the center by half atomic distance to 2. NN.")
        print(f"{'Atom 1':>20}:", atom1)
        print(f"{'Atom 2':>20}:", atom2)
        center += np.array([0, NN_2_DIST/2, 0])  # shift the center by half the distance to the 2nd NN
        print(f"{'New Center':>20}:", center)

    return atom_idx, center


def count_removals(func):
    """Decorator to count number of atoms before and after removing overlaps."""
    def wrapper(*args, **kwargs):
        initial_count = len(args[0])
        result = func(*args, **kwargs)
        final_count = len(result)
        print(f"       Initial number of atoms: {initial_count}")
        print(f" Atoms after removing overlaps: {final_count}")
        return result
    return wrapper

# ---------------------------
# Structure generation
# ---------------------------
def find_overlap(structure, tol=0.1):
    """Find overlapping atoms in a structure."""
    coords = get_coordinates(structure)
    tree = cKDTree(coords)
    pairs = tree.query_pairs(r=tol)  # find pairs of atoms closer than `tol`
    # print(f"Found {len(pairs)} overlapping pairs of atoms.")
    return pairs

@count_removals
def delete_overlaps(structure, pairs=None):
    """Delete overlapping atoms from a structure."""
    if pairs is None:
        overlaps = find_overlap(structure)
    else:
        overlaps = pairs
    to_delete = set()
    for i, j in overlaps:
        to_delete.add(j)  # arbitrarily delete the second atom in each pair
    mask = np.array([i not in to_delete for i in range(len(structure))])
    return structure.sub(mask)


def generate_structure(width=3, length=5, bond=1.42, kind='armchair', vacuum=3.0):
    """Generate 3 rotated nanoribbons stacked around a central hexagon."""
    base = sisl.geom.graphene_nanoribbon(width=width, bond=bond, kind=kind, vacuum=vacuum)
    base = base.repeat(length, axis=0)
    
    # find central hexagon
    _, rotation_origin = guess_hexagon_center(base)
    
    # rotate and add ribbons
    structure = base.copy()
    for i in range(1, 3):
        angle = i * 60
        rotated = base.rotate(angle=angle, v=[0, 0, 1], origin=rotation_origin)
        structure += rotated
        
    overlaps = find_overlap(structure)
    indexes = {}
    for i in range(3):
        indexes[i] = list(range(i * len(base), (i + 1) * len(base)))
    if overlaps:
        print(f"Found {len(overlaps)} overlapping pairs of atoms before removing overlaps.")
        for key in indexes:
            for i, j in overlaps:
                if j in indexes[key]:
                    indexes[key].remove(j)
        arm1 = len(indexes[0])
        arm2 = len(indexes[1])
        arm3 = len(indexes[2])
        indexes[1] = list(range(arm1, arm1 + arm2))
        indexes[2] = list(range(arm1 + arm2, arm1 + arm2 + arm3))
    return delete_overlaps(structure), indexes


# ---------------------------
# Plotting utilities
# ---------------------------

def plot_with_center(structure, **KWARGS):
    """Plot structure and highlight central hexagon + geometric center."""
    
    KWARGS.setdefault("axes", "xy")
    KWARGS.setdefault("bind_bonds_to_ats", True)
    atom_idx, hex_center = guess_hexagon_center(structure)
    # print(f"Hexagon atom indices: {atom_idx}")
    geom_center = structure.center()

    fig = structure.plot(**KWARGS)

    # highlight hexagon atoms
    fig.update_inputs(atoms_style={"color": "red", "atoms": atom_idx.tolist()})

    # add markers
    fig.add_trace(go.Scatter(
        x=[geom_center[0]], y=[geom_center[1]],
        mode="markers", marker=dict(size=10, color="red", symbol="x"),
        name="Geometric Center"
    ))
    
    fig.add_trace(go.Scatter(
        x=[hex_center[0]], y=[hex_center[1]],
        mode="markers", marker=dict(size=10, color="blue", symbol="circle"),
        name="Rotation Center"
    ))

    return fig

WIDTH = 20
LENGTH = 100
before = 270
after  = 232
graphene, atom_index = generate_structure(width=WIDTH, length=LENGTH)

fig = plot_with_center(graphene)

fig.update_inputs(atoms_style=[
    {"color": "red", "atoms": atom_index[0]},
    {"color": "blue", "atoms": atom_index[1]},
    {"color": "green", "atoms": atom_index[2]}
    ])
fig.show()


Found 800 overlapping pairs of atoms before removing overlaps.
       Initial number of atoms: 12000
 Atoms after removing overlaps: 11400
